# 04. Build Final CULane Dataset from Usable Auto Labels

이 노트북의 목적은 03 리뷰 큐에서 `usable_auto`로 판정된 field1+field2 라벨만 사용해 CLRKDNet 공식 `CULane` dataloader가 바로 읽을 수 있는 최종 학습 데이터셋을 만드는 것이다.

이번 v1 정책은 일부 밝은 샘플을 억지로 살리지 않는다. 라벨 recall보다 precision을 우선한다. 따라서 `review_needed`와 `reject`는 학습에서 제외하고, 깨끗한 baseline을 먼저 만든다.

## 공식 CULane 구조로 맞출 항목

CLRKDNet의 `clrkd/datasets/culane.py`는 다음 구조를 기대한다.

```text
dataset_root/
├─ list/
│  ├─ train_gt.txt
│  ├─ val.txt
│  ├─ test.txt
│  └─ test_split/
│     ├─ test0_normal.txt
│     └─ test1_crowd.txt ... test8_night.txt
├─ driver_.../
│  ├─ frame.jpg
│  └─ frame.lines.txt
└─ laneseg_label_w16/
   └─ driver_.../
      └─ frame.png
```

디스크에 저장하는 `.lines.txt`와 mask는 raw image 좌표계인 `1296x972` 기준이다. `cut_height=445`, resize `800x320`, `/255` 정규화는 학습 dataloader/config 단계에서 처리한다.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import json
import math
import os
import shutil
import time

import cv2
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(r'~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization')
EXP_ROOT = PROJECT_ROOT / '10_experiments' / '12_clrkdnet_supervised_rebuild'
REVIEW_ROOT = EXP_ROOT / 'review_outputs' / '03_review_q'
ASSET_LANE_ROOT = PROJECT_ROOT / '20_shared_assets' / 'dataset' / 'lane'
DATASET_NAME = 'map_culane_localfit_train_field1_field2_v1'
DATASET_ROOT = ASSET_LANE_ROOT / DATASET_NAME

MANIFEST_PATH = REVIEW_ROOT / 'label_candidate_manifest_field12.csv'
LINES_PATH = REVIEW_ROOT / 'selected_lines_field12.csv'

RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
MODEL_W = 800
MODEL_H = 320
NUM_POINTS = 72
MAX_LANES = 4
MASK_THICKNESS = 10
IMAGE_BASE = 'drv_f12_v1'
MASK_BASE = 'laneseg_label_w16'
TRAIN_RATIO = 0.8

print('manifest:', MANIFEST_PATH, MANIFEST_PATH.exists())
print('lines:', LINES_PATH, LINES_PATH.exists())
print('dataset root:', DATASET_ROOT)
assert MANIFEST_PATH.exists(), MANIFEST_PATH
assert LINES_PATH.exists(), LINES_PATH

## 1. 03 리뷰 큐 결과에서 `usable_auto`만 선택

`review_needed`에는 hard case도 있지만 라벨 품질이 섞여 있다. 이번 v1 baseline에서는 학습 실패 원인을 분리하기 위해 `usable_auto`만 사용한다.

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
lines_df = pd.read_csv(LINES_PATH)

usable_df = manifest_df[manifest_df['status'] == 'usable_auto'].copy().reset_index(drop=True)
excluded_df = manifest_df[manifest_df['status'] != 'usable_auto'].copy().reset_index(drop=True)
usable_lines_df = lines_df[lines_df['frame_status'] == 'usable_auto'].copy().reset_index(drop=True)

print('all frames:', len(manifest_df))
print('usable_auto frames:', len(usable_df))
print('excluded frames:', len(excluded_df))
print('usable lines:', len(usable_lines_df))
print('status counts:')
display(manifest_df['status'].value_counts().rename_axis('status').reset_index(name='count'))
print('usable selected_count counts:')
display(usable_df['selected_count'].value_counts().sort_index().rename_axis('selected_count').reset_index(name='count'))
print('group counts:')
display(usable_df['group'].value_counts().rename_axis('group').reset_index(name='count'))

## 2. Scene-wise chronological train/val split

예전 파이프라인의 알파벳순 split은 특정 scene이 val에 쏠릴 수 있었다. 여기서는 `field + parent directory`를 scene key로 보고, 각 scene 내부에서 시간순 앞쪽 80%를 train, 뒤쪽 20%를 val로 둔다. 이렇게 하면 field1/field2와 각 주행 상황이 train/val에 모두 남는다.

In [ ]:
def scene_key_from_rel(rel: str) -> str:
    p = Path(str(rel).replace('\\', '/'))
    # rel = field1/lane/intersection/file.jpg -> field1/lane/intersection
    return '/'.join(p.parts[:-1]) if len(p.parts) > 1 else p.parts[0]

usable_df['scene_key'] = usable_df['rel'].map(scene_key_from_rel)
usable_df = usable_df.sort_values(['scene_key', 'rel']).reset_index(drop=True)

split_rows = []
for scene_key, group_df in usable_df.groupby('scene_key', sort=True):
    g = group_df.sort_values('rel').copy().reset_index(drop=True)
    n = len(g)
    if n <= 1:
        val_count = 0
    else:
        val_count = max(1, int(round(n * (1.0 - TRAIN_RATIO))))
        if val_count >= n:
            val_count = n - 1
    split = ['train'] * (n - val_count) + ['val'] * val_count
    g['split'] = split
    split_rows.append(g)

split_df = pd.concat(split_rows, ignore_index=True)
print('split counts:')
display(split_df['split'].value_counts().rename_axis('split').reset_index(name='count'))
print('group x split:')
display(pd.crosstab(split_df['group'], split_df['split']))
print('top scenes:')
display(split_df.groupby(['scene_key', 'split']).size().unstack(fill_value=0).sort_values(['train','val'], ascending=False).head(30))

## 3. CULane 파일 생성

각 usable frame에 대해 다음 파일을 생성한다.

- 이미지 복사: `drv_f12_v1/.../*.jpg`
- 같은 위치의 `.lines.txt`: raw 좌표 점 시퀀스
- segmentation mask: `laneseg_label_w16/drv_f12_v1/.../*.png`
- list 파일: `train_gt.txt`, `val.txt`, `test.txt`, `test_split/test0_normal.txt`

mask는 `0=background`, `1..N=lane index`로 저장한다. 이번 local-fit 정책은 최대 2개 lane만 생성하지만 공식 config와 호환되도록 exist flag는 4칸을 유지한다.

In [ ]:
def win_long(path: Path) -> str:
    s = str(path)
    if os.name == 'nt':
        s = os.path.abspath(s)
        if not s.startswith('\\\\?\\'):
            s = '\\\\?\\' + s
    return s


def safe_scene(scene_key: str) -> str:
    scene = scene_key.replace('\\', '/').replace('/', '_')
    clean = ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in scene)
    digest = hashlib.md5(scene_key.encode('utf-8')).hexdigest()[:6]
    return f'{digest}_{clean[:42]}'


def safe_file_name(rel: str, stem: str) -> str:
    digest = hashlib.md5(str(rel).replace('\\', '/').encode('utf-8')).hexdigest()[:8]
    clean = ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in str(stem))
    return f'{digest}_{clean[:48]}.jpg'


def parse_points_text(text: str):
    vals = [float(x) for x in str(text).strip().split()]
    pts = []
    for i in range(0, len(vals) - 1, 2):
        x, y = vals[i], vals[i + 1]
        if math.isfinite(x) and math.isfinite(y) and 0 <= x < RAW_W and 0 <= y < RAW_H:
            pts.append((float(x), float(y)))
    # CULane loader sorts by y later, but keep bottom->top convention in file.
    pts = sorted(set((round(x, 3), round(y, 3)) for x, y in pts), key=lambda p: p[1], reverse=True)
    return pts


def line_text_from_points(points):
    return ' '.join(f'{x:.3f} {y:.3f}' for x, y in points)


def lane_exist_flags(num_lanes: int):
    return [1 if i < num_lanes else 0 for i in range(MAX_LANES)]


def draw_lane_mask(shape, lanes, thickness=10):
    mask = np.zeros(shape, dtype=np.uint8)
    for idx, lane in enumerate(lanes, start=1):
        if idx > MAX_LANES or len(lane) < 2:
            continue
        pts = np.array([[int(round(x)), int(round(y))] for x, y in lane], dtype=np.int32)
        cv2.polylines(mask, [pts], isClosed=False, color=int(idx), thickness=int(thickness), lineType=cv2.LINE_AA)
    return mask


def ensure_parent(path: Path):
    os.makedirs(win_long(path.parent), exist_ok=True)


def write_text(path: Path, text: str):
    ensure_parent(path)
    with open(win_long(path), 'w', encoding='utf-8', newline='') as f:
        f.write(text)


def copy_file(src: Path, dst: Path):
    ensure_parent(dst)
    shutil.copy2(win_long(src), win_long(dst))


def save_mask(path: Path, mask: np.ndarray):
    ensure_parent(path)
    ok, encoded = cv2.imencode('.png', mask)
    if not ok:
        raise RuntimeError(f'failed to encode mask: {path}')
    with open(win_long(path), 'wb') as f:
        f.write(encoded.tobytes())

line_groups = defaultdict(list)
for _, row in usable_lines_df.iterrows():
    line_groups[str(row['rel'])].append(row)

# Rebuild cleanly. This deletes only this generated dataset folder.
if DATASET_ROOT.exists():
    shutil.rmtree(win_long(DATASET_ROOT))
(DATASET_ROOT / 'list').mkdir(parents=True, exist_ok=True)

records = []
start = time.time()

for idx, row in split_df.iterrows():
    rel = str(row['rel'])
    src = Path(str(row['abs_path']))
    if not src.exists():
        raise FileNotFoundError(src)

    scene_key = str(row['scene_key'])
    scene = safe_scene(scene_key)
    filename = safe_file_name(rel, row['stem'])
    img_rel = Path(IMAGE_BASE) / str(row['group']) / scene / filename
    mask_rel = Path(MASK_BASE) / img_rel.with_suffix('.png')
    img_dst = DATASET_ROOT / img_rel
    lines_dst = img_dst.with_suffix('.lines.txt')
    mask_dst = DATASET_ROOT / mask_rel

    lane_rows = sorted(line_groups.get(rel, []), key=lambda r: int(r.get('lane_idx', 0)))
    lanes = []
    for lr in lane_rows:
        pts = parse_points_text(lr['points_text'])
        if len(pts) > 2:
            lanes.append(pts)
    if not lanes:
        raise RuntimeError(f'usable row has no valid lane points: {rel}')
    lanes = lanes[:MAX_LANES]

    copy_file(src, img_dst)
    write_text(lines_dst, '\n'.join(line_text_from_points(lane) for lane in lanes) + '\n')
    mask = draw_lane_mask((RAW_H, RAW_W), lanes, thickness=MASK_THICKNESS)
    save_mask(mask_dst, mask)

    records.append({
        'split': row['split'],
        'group': row['group'],
        'scene_key': scene_key,
        'source_rel': rel,
        'source_abs_path': str(src),
        'dataset_image': img_rel.as_posix(),
        'dataset_mask': mask_rel.as_posix(),
        'num_lanes': len(lanes),
        'selected_count': int(row['selected_count']),
        'candidate_count': int(row['candidate_count']),
        'diagnostic_flags': row.get('diagnostic_flags', ''),
        'policy': row.get('policy', ''),
    })

print('built records:', len(records), 'elapsed_min:', round((time.time() - start) / 60, 2))
print('dataset root:', DATASET_ROOT)

## 4. list 파일과 manifest 저장

`train_gt.txt`는 이미지, mask, lane exist flag를 포함한다. `val.txt`와 `test.txt`는 공식 CULane처럼 이미지 경로만 둔다. 평가 카테고리 파일은 normal만 val과 동일하게 채우고 나머지는 빈 파일로 둔다.

In [ ]:
def list_image_line(record):
    return '/' + record['dataset_image']


def list_train_line(record):
    flags = ' '.join(map(str, lane_exist_flags(int(record['num_lanes']))))
    return f"/{record['dataset_image']} /{record['dataset_mask']} {flags}"

train_records = [r for r in records if r['split'] == 'train']
val_records = [r for r in records if r['split'] == 'val']

list_dir = DATASET_ROOT / 'list'
write_text(list_dir / 'train_gt.txt', '\n'.join(list_train_line(r) for r in train_records) + '\n')
write_text(list_dir / 'val.txt', '\n'.join(list_image_line(r) for r in val_records) + '\n')
write_text(list_dir / 'test.txt', '\n'.join(list_image_line(r) for r in val_records) + '\n')

split_dir = list_dir / 'test_split'
split_dir.mkdir(parents=True, exist_ok=True)
write_text(split_dir / 'test0_normal.txt', '\n'.join(list_image_line(r) for r in val_records) + '\n')
for i, name in enumerate(['crowd', 'hlight', 'shadow', 'noline', 'arrow', 'curve', 'cross', 'night'], start=1):
    write_text(split_dir / f'test{i}_{name}.txt', '')

# Dataset manifest.
manifest_out = DATASET_ROOT / 'build_manifest.csv'
with open(win_long(manifest_out), 'w', encoding='utf-8-sig', newline='') as f:
    fieldnames = list(records[0].keys()) if records else []
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

# Excluded metadata only, no images copied.
excluded_out = DATASET_ROOT / 'excluded_manifest.csv'
excluded_df.to_csv(excluded_out, index=False, encoding='utf-8-sig')

summary = {
    'name': DATASET_NAME,
    'dataset_root': str(DATASET_ROOT),
    'source_review_root': str(REVIEW_ROOT),
    'source_manifest': str(MANIFEST_PATH),
    'source_lines': str(LINES_PATH),
    'policy': {
        'accept_status': 'usable_auto only',
        'excluded_statuses': sorted(excluded_df['status'].dropna().unique().tolist()),
        'hsv': {'lower': [20, 140, 70], 'upper': [42, 255, 255]},
        'morphology': [{'op': 'open', 'kernel': 3}, {'op': 'close', 'kernel': 3}],
        'local_fit': {'row_mode': 'inner_edge', 'window_px': 120, 'max_lanes_per_frame': 2},
    },
    'geometry': {
        'raw_width': RAW_W,
        'raw_height': RAW_H,
        'cut_height': CUT_HEIGHT,
        'model_width': MODEL_W,
        'model_height': MODEL_H,
        'num_points': NUM_POINTS,
        'max_lanes': MAX_LANES,
        'sample_y': 'range(971, 444, -20)',
    },
    'counts': {
        'input_frames': int(len(manifest_df)),
        'usable_auto_frames': int(len(usable_df)),
        'excluded_frames': int(len(excluded_df)),
        'built_frames': int(len(records)),
        'train_rows': int(len(train_records)),
        'val_rows': int(len(val_records)),
        'test_rows': int(len(val_records)),
        'selected_lines': int(sum(r['num_lanes'] for r in records)),
    },
    'status_counts_source': {str(k): int(v) for k, v in manifest_df['status'].value_counts().to_dict().items()},
    'split_counts': {str(k): int(v) for k, v in pd.Series([r['split'] for r in records]).value_counts().to_dict().items()},
    'group_split_counts': {
        f'{g}::{s}': int(v)
        for (g, s), v in pd.DataFrame(records).groupby(['group', 'split']).size().to_dict().items()
    },
}
write_text(DATASET_ROOT / 'build_summary.json', json.dumps(summary, ensure_ascii=False, indent=2))

print(json.dumps(summary['counts'], ensure_ascii=False, indent=2))
print('saved manifest:', manifest_out)
print('saved excluded manifest:', excluded_out)
print('saved summary:', DATASET_ROOT / 'build_summary.json')

## 5. 구조 검증

공식 dataloader가 읽기 전에 다음을 확인한다.

- list 파일 row 수
- 이미지와 `.lines.txt` 존재 여부
- train mask 존재 여부
- 이미지/mask 크기 `1296x972`
- mask 값이 `0..4` 범위 안인지

In [ ]:
def path_exists(path: Path) -> bool:
    return os.path.exists(win_long(path))


def read_text_safe(path: Path) -> str:
    with open(win_long(path), 'r', encoding='utf-8') as f:
        return f.read()


def read_image_safe(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(win_long(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def read_list(path: Path):
    return [line.strip().split() for line in read_text_safe(path).splitlines() if line.strip()]

issues = []
counts = {}
for name in ['train_gt.txt', 'val.txt', 'test.txt']:
    rows = read_list(DATASET_ROOT / 'list' / name)
    counts[name] = len(rows)
    for row_idx, parts in enumerate(rows):
        img_rel = parts[0].lstrip('/')
        img_path = DATASET_ROOT / img_rel
        lines_path = img_path.with_suffix('.lines.txt')
        if not path_exists(img_path):
            issues.append(f'{name}:{row_idx} missing image {img_rel}')
            continue
        img = read_image_safe(img_path, cv2.IMREAD_COLOR)
        if img is None:
            issues.append(f'{name}:{row_idx} unreadable image {img_rel}')
        elif img.shape[:2] != (RAW_H, RAW_W):
            issues.append(f'{name}:{row_idx} bad image shape {img.shape[:2]} {img_rel}')
        if not path_exists(lines_path):
            issues.append(f'{name}:{row_idx} missing lines {lines_path.relative_to(DATASET_ROOT)}')
        elif not read_text_safe(lines_path).strip():
            issues.append(f'{name}:{row_idx} empty lines {lines_path.relative_to(DATASET_ROOT)}')
        if name == 'train_gt.txt':
            if len(parts) != 2 + MAX_LANES:
                issues.append(f'{name}:{row_idx} bad train row field count {len(parts)}')
                continue
            mask_rel = parts[1].lstrip('/')
            mask_path = DATASET_ROOT / mask_rel
            if not path_exists(mask_path):
                issues.append(f'{name}:{row_idx} missing mask {mask_rel}')
                continue
            mask = read_image_safe(mask_path, cv2.IMREAD_UNCHANGED)
            if mask is None:
                issues.append(f'{name}:{row_idx} unreadable mask {mask_rel}')
            elif mask.shape[:2] != (RAW_H, RAW_W):
                issues.append(f'{name}:{row_idx} bad mask shape {mask.shape[:2]} {mask_rel}')
            else:
                values = set(int(x) for x in np.unique(mask))
                if not values.issubset(set(range(MAX_LANES + 1))):
                    issues.append(f'{name}:{row_idx} bad mask values {sorted(values)} {mask_rel}')

validation_summary = {
    'dataset_root': str(DATASET_ROOT),
    'counts': counts,
    'issue_count': len(issues),
    'issues_sample': issues[:100],
}
write_text(DATASET_ROOT / 'validation_summary.json', json.dumps(validation_summary, ensure_ascii=False, indent=2))
print(json.dumps(validation_summary, ensure_ascii=False, indent=2))
assert not issues, issues[:10]


## 6. 다음 단계

이 데이터셋은 `usable_auto only` baseline이다. 다음 05 노트북에서는 이 데이터셋을 Colab에 올려 supervised fine-tuning을 수행한다. 그때 반드시 확인할 것은 학습 입력과 추론 입력의 정합성이다.

- 학습 dataloader 입력: BGR float32 `[0, 1]`
- export/runtime 입력: 같은 crop/resize/order/range로 맞춤
- 공식 후처리: softmax threshold → LineIoU NMS → top-k

이 세 가지를 stop-check로 박아두고 넘어간다.